# MVELSA — Pipeline Completo (Google Colab)

Execute as células **em ordem**. Cada seção corresponde a uma etapa do pipeline:

1. Configuração do ambiente
2. Criação do dataset (crops fullHD633)
3. Treino dos especialistas MVELSA (ae_times=3 — aproveita a VRAM do Colab)
4. Geração das features latentes
5. Treino do classificador
6. Calibração + extração REP + meta-classificador
7. Validação cega comparativa


## ⚙️ Célula 1 — Montar Drive e configurar caminhos
**Ajuste `FULLHD_DIR` se necessário.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# =============================================================
# CONFIGURE OS CAMINHOS AQUI
# =============================================================
PROJECT_ROOT = "/content/drive/MyDrive/seadev HD/MVELSA"

# Pasta com as imagens fullHD633 e _annotations.coco.json
FULLHD_DIR   = f"{PROJECT_ROOT}/fullHD633/fullHD"

# Onde os crops serão salvos (dentro do Drive para persistir entre sessões)
DATA_OUT     = f"{PROJECT_ROOT}/data/coco_cropped"
# =============================================================

V2_DIR = f"{PROJECT_ROOT}/experiments/images/coco_surface/v2_cropped_optimized"
V3_DIR = f"{PROJECT_ROOT}/experiments/images/coco_surface/v3_detection_pipeline"

# Adicionar raiz do projeto e V2 ao path (imports dos módulos elsanet/data)
for p in [PROJECT_ROOT, V2_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"FULLHD_DIR   : {FULLHD_DIR}")
print(f"DATA_OUT     : {DATA_OUT}")
print(f"Existe FULLHD_DIR? {os.path.exists(FULLHD_DIR)}")
print(f"Existe V2_DIR?     {os.path.exists(V2_DIR)}")

## ⚙️ Célula 2 — Verificar GPU e dependências

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Instalar dependências que possam faltar no Colab
!pip install -q scikit-learn pandas tqdm matplotlib pillow

## 📦 Etapa 1 — Criar dataset de crops (fullHD633)

Gera `train/valid/test` com crops das 5 classes (BOAT, BUOY, LAND, SHIP, SKY).  
**Pule esta célula** se o dataset já existir em `DATA_OUT`.

In [ ]:
import os, sys
os.chdir(V2_DIR)

import json
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# --- Configuração ---
SOURCE_DIR         = FULLHD_DIR
ANNOT_FILE         = os.path.join(SOURCE_DIR, '_annotations.coco.json')
OUT_DIR            = DATA_OUT
FOCUS_CLASSES      = {1, 3, 4, 5, 6}      # BOAT, BUOY, LAND, SHIP, SKY
BACKGROUND_CLASSES = {4, 6}               # LAND, SKY — zona pura
MIN_BBOX_AREA      = 1600                 # 40×40 px mínimo
IOU_THRESHOLD      = 0.50
MAX_CROPS_PER_IMAGE_CLASS = 4
MAX_SAMPLES_PER_CLASS = {3: 400, 6: 200}

def _iou(b1, b2):
    ax1,ay1 = b1[0],b1[1]; ax2,ay2 = ax1+b1[2],ay1+b1[3]
    bx1,by1 = b2[0],b2[1]; bx2,by2 = bx1+b2[2],by1+b2[3]
    ix1,iy1 = max(ax1,bx1),max(ay1,by1); ix2,iy2 = min(ax2,bx2),min(ay2,by2)
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1); union=b1[2]*b1[3]+b2[2]*b2[3]-inter
    return inter/union if union>0 else 0.0

def _deduplicate(df):
    keep = []
    for (img_path, cat_id), group in df.groupby(['abs_path','cat_id']):
        if cat_id not in BACKGROUND_CLASSES:
            keep.extend(group.index.tolist()); continue
        group = group.copy()
        group['_area'] = group['bbox'].apply(lambda b: b[2]*b[3])
        group = group.sort_values('_area', ascending=False)
        accepted = []
        for idx, row in group.iterrows():
            if not any(_iou(row['bbox'], group.loc[k,'bbox'])>IOU_THRESHOLD for k in accepted):
                accepted.append(idx)
            if len(accepted)>=MAX_CROPS_PER_IMAGE_CLASS: break
        keep.extend(accepted)
    return df.loc[keep].drop(columns=['_area'], errors='ignore')

def _safe_zone(bbox, other_bboxes, min_side=40):
    if not other_bboxes: return bbox
    x,y,w,h = [float(v) for v in bbox]; x2,y2 = x+w,y+h
    forbidden = [(oy,oy+oh) for ob in other_bboxes
                 for ox,oy,ow,oh in [[float(v) for v in ob]]
                 if (ox+ow)>x and ox<x2]
    if not forbidden: return bbox
    min_fy=min(fy for fy,_ in forbidden); max_fy=max(fy2 for _,fy2 in forbidden)
    top_h=min_fy-y; bot_h=y2-max_fy
    if top_h>=bot_h and top_h>=min_side: return [x,y,w,top_h]
    elif bot_h>=min_side: return [x,max_fy,w,bot_h]
    elif top_h>=min_side: return [x,y,w,top_h]
    return None

def create_cropped_dataset():
    print(f'Lendo anotações: {ANNOT_FILE}')
    with open(ANNOT_FILE) as f: data = json.load(f)
    cat_name = {c['id']:c['name'] for c in data['categories']}
    img_info  = {img['id']:img['file_name'] for img in data['images']}

    samples = {}
    for ann in data['annotations']:
        cid = ann['category_id']
        if cid not in FOCUS_CLASSES: continue
        bbox = ann['bbox']
        if bbox[2]*bbox[3] < MIN_BBOX_AREA: continue
        aid = ann['id']
        if aid not in samples:
            samples[aid] = {'abs_path': os.path.join(SOURCE_DIR, img_info[ann['image_id']]),
                            'bbox': bbox, 'cat_id': cid, 'cat_name': cat_name[cid], 'ann_id': aid}

    df = pd.DataFrame(list(samples.values()))
    print(f'Anotações viáveis: {len(df)}')
    before = len(df); df = _deduplicate(df)
    print(f'Após deduplicação IoU: {len(df)} ({before-len(df)} removidas)')

    frames = []
    for cid, group in df.groupby('cat_id'):
        cap = MAX_SAMPLES_PER_CLASS.get(cid)
        if cap and len(group)>cap:
            group = group.sample(n=cap, random_state=42)
            print(f'  [cap] {cat_name[cid]}: {len(group)} amostras')
        frames.append(group)
    df = pd.concat(frames).reset_index(drop=True)

    print('\nDistribuição por classe:')
    for cid, cnt in df['cat_id'].value_counts().sort_index().items():
        print(f'  {cat_name[cid]:10s} (id {cid}): {cnt}')

    unique_imgs = df[['abs_path']].drop_duplicates()
    img_cls = df.groupby('abs_path')['cat_id'].first().reset_index()
    unique_imgs = unique_imgs.merge(img_cls, on='abs_path')

    def safe_split(data, test_size, col):
        if (data[col].value_counts()<2).any():
            return train_test_split(data, test_size=test_size, random_state=42)
        return train_test_split(data, test_size=test_size, stratify=data[col], random_state=42)

    train_imgs, temp = safe_split(unique_imgs, 0.30, 'cat_id')
    val_imgs, test_imgs = safe_split(temp, 0.50, 'cat_id')
    split_map = {'train': df[df['abs_path'].isin(train_imgs['abs_path'])],
                 'valid': df[df['abs_path'].isin(val_imgs['abs_path'])],
                 'test':  df[df['abs_path'].isin(test_imgs['abs_path'])]}

    img_other = {}
    for _, row in df.iterrows():
        img_other.setdefault(row['abs_path'], {}).setdefault(row['cat_id'], []).append(row['bbox'])

    os.makedirs(OUT_DIR, exist_ok=True)
    for split_name, df_split in split_map.items():
        split_dir = os.path.join(OUT_DIR, split_name)
        os.makedirs(split_dir, exist_ok=True)
        csv_data, skipped = [], 0
        print(f'\nProcessando {split_name}: {len(df_split)} objetos...')

        for _, row in df_split.iterrows():
            try:
                cid = row['cat_id']; img_path = row['abs_path']; bbox = row['bbox']
                orig_cy_norm = round((row['bbox'][1]+row['bbox'][3]/2) /
                                     Image.open(img_path).size[1], 4)

                if cid in BACKGROUND_CLASSES:
                    others = [b for c,bboxes in img_other.get(img_path,{}).items()
                              if c!=cid for b in bboxes]
                    bbox = _safe_zone(bbox, others)
                    if bbox is None: skipped+=1; continue

                img = Image.open(img_path).convert('RGB')
                iw, ih = img.size
                x, y, w, h = [int(v) for v in bbox]
                orig_cy_norm = round((y+h/2)/ih, 4)

                side = min(w, max(h, h*2)) if cid==5 else min(w, h)
                cx=x+w//2; cy=y+h//2
                x1=max(0,cx-side//2); y1=max(0,cy-side//2)
                x2=min(iw,x1+side);  y2=min(ih,y1+side)
                if x2-x1<5 or y2-y1<5: continue

                crop = img.crop((x1,y1,x2,y2)).resize((64,64), Image.LANCZOS)
                fname = f"{row['cat_name']}_{row['ann_id']}.jpg"
                crop.save(os.path.join(split_dir, fname))
                csv_data.append({'filename':fname,'class_id':cid,
                                 'class_name':row['cat_name'],'cy_norm':orig_cy_norm})
            except Exception as e:
                print(f'  Erro: {e}')

        if skipped: print(f'  [!] {skipped} LAND/SKY descartados (zona pura insuficiente)')
        df_out = pd.DataFrame(csv_data)
        df_out.to_csv(os.path.join(split_dir,'labels.csv'), index=False)
        print(f'  Salvos: {len(df_out)} crops')
        for cid, cnt in df_out['class_id'].value_counts().sort_index().items():
            print(f'    {cat_name.get(cid,cid):10s}: {cnt}')

create_cropped_dataset()
print('\nDataset criado com sucesso!')

## 🧠 Etapa 2 — Treino MVELSA (ae_times=3, batch=128)

Com GPU T4/V100 do Colab temos VRAM suficiente para `ae_times=3` (impossível localmente com 4 GB).

In [ ]:
import os, sys, collections, torch
import torchvision.transforms as transforms
os.chdir(V2_DIR)

from elsanet.mvelsa import MVELSA, RMSELoss
from data.data_preparation import DataPreparation
from cropped_data_generator import CroppedDataset, PadToSquare

resolution = (64, 64)
channels   = 3

transform_train = transforms.Compose([
    PadToSquare(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Resize(size=resolution),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
transform_val = transforms.Compose([
    PadToSquare(),
    transforms.ToTensor(),
    transforms.Resize(size=resolution),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

DATA_CROPPED = f"{DATA_OUT}"  # absolute path evita problema de cwd
dataset_train = CroppedDataset(DATA_CROPPED, train=True,  transform=transform_train)
dataset_val   = CroppedDataset(DATA_CROPPED, train=False, transform=transform_val)

focus_classes = {1, 3, 4, 5, 6}
category_ids  = [c for c in set(dataset_train.targets).intersection(set(dataset_val.targets))
                 if c in focus_classes]
print(f'Classes: {sorted(category_ids)}')

train_counts = collections.Counter(dataset_train.targets)
val_counts   = collections.Counter(dataset_val.targets)
for cid in sorted(category_ids):
    print(f'  Class {cid}: train={train_counts.get(cid,0)}, val={val_counts.get(cid,0)}')

data_parameters = {
    'data_type': 'image',
    'file_path': DATA_CROPPED + '/',
    'dataset_name': CroppedDataset,
    'transform': transform_val,
    'batch_size': 128,
    'data_train_lenght': None,
    'data_val_lenght': 50,
}

data_instance = DataPreparation(data_parameters).gen()
data_instance.train = dataset_train
data_instance.test  = dataset_val
data_instance.get_labels(category_ids)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

model_hyperparameters = {
    'ae_architecture': [64*64*channels, 1024, 256, 128],
    'ae_times': 3,        # Colab tem VRAM suficiente para ae_times=3
    'activation': 'ReLU',
    'epochs': 100,
    'learning_rate': 0.001,
    'loss_function': RMSELoss().to(device),
    'seed': 42,
    'device': device,
}

mvelsa = MVELSA(model_hyperparameters)
print('INICIANDO TREINAMENTO MVELSA')
mvelsa.fit(data_instance)
print('TREINAMENTO FINALIZADO')

mvelsa.save(file_name='ELSA_MODEL_CROPPED_SURFACE')
print('Modelo salvo: ELSA_MODEL_CROPPED_SURFACE')

## 🔢 Etapa 3 — Gerar features latentes (encoded data)

In [ ]:
import os, sys
os.chdir(V2_DIR)
# Garante que o caminho absoluto do dataset é usado
import cropped_data_generator as cdg
# Sobrescreve a constante de focus_classes no módulo gen_cropped_encoded antes do %run
%run gen_cropped_encoded.py

## 🎯 Etapa 4 — Treino do classificador (100 épocas)

In [ ]:
import os
os.chdir(V2_DIR)
%run train_cropped_classifier.py

## 📐 Etapa 5 — Calibração de especialistas

In [ ]:
import os
os.chdir(V3_DIR)
%run calibrate_experts.py

## 🔍 Etapa 6 — Extração de perfis REP (50D + cy_norm)

In [ ]:
import os
os.chdir(V3_DIR)
%run extract_rep_profiles.py

## 🌲 Etapa 7 — Meta-classificador REP (Random Forest)

In [ ]:
import os
os.chdir(V3_DIR)
%run train_meta_classifier.py

## ✅ Etapa 8 — Validação Cega Comparativa (Estratégias B, C, D)

In [ ]:
import os
os.chdir(V3_DIR)
%run validate_v2_blind.py

---
## 💾 (Opcional) Copiar modelos treinados de volta para o Drive

Os modelos são salvos diretamente no Drive via `V2_DIR` e `V3_DIR`.  
Se estiver usando `/content/` local, execute a célula abaixo para copiar.

In [ ]:
# Execute SOMENTE se os modelos foram salvos em /content/ (fora do Drive)
# e você quer persistir no Drive.

# import shutil
# for fname in ['ELSA_MODEL_CROPPED_SURFACE', 'MVELSA_CLASSIFIER.pth',
#               'ENCODED_DATA_CROPPED_SURFACE', 'Cropped_Loss_Graph.png']:
#     src = os.path.join(V2_DIR, fname)
#     if os.path.exists(src):
#         shutil.copy(src, V2_DIR)  # já está no Drive se V2_DIR aponta para o Drive
#         print(f'Copiado: {fname}')

print('Modelos salvos em:', V2_DIR)
print('Artefatos V3 em: ', V3_DIR)